# Example of Optax Optimizer State Sharding Inference for Cross-Compiling

This notebook demonstrates an end-to-end example of using optax optimizer state sharding inference in cross-compilation. Note that we only use 8 virtual JAX CPU devices throughout (no physical TPUs required):

1. **Zero-Allocation Abstract Model Setup:** Build a 2D `AxisType.Auto` `('data', 'model')` mesh and a Transformer PyTree (`2D` embeddings/MLPs, `3D` multi-head attention weights, and `1D` layer-norm scales) using `jax.ShapeDtypeStruct` + `NamedSharding`.
2. **`jax.eval_shape` vs. `optax.tree_utils.abstract_init`:** Show how naive `jax.eval_shape(tx.init, abstract_params)` silently strips `Auto` shardings and replicates `mu` and `nu` on every chip ($8\times$ optimizer memory), whereas `optax.tree_utils.abstract_init(tx, abstract_params)` preserves exact `('data', 'model')` shardings.

In [1]:
import math
from typing import Any
import jax

# Simulate an 8-chip accelerator topology (2x4) on CPU
jax.config.update('jax_num_cpu_devices', 8)

import jax.numpy as jnp
from jax.sharding import AxisType, NamedSharding, PartitionSpec as P
import optax

print('JAX version:', jax.__version__, '| Virtual devices:', len(jax.devices()))

JAX version: 0.11.2 | Virtual devices: 8


## 1: Build Abstract Transformer `params` & `batch` on an `AxisType.Auto` Mesh (0 Bytes Allocated)

We define a 2D `('data', 'model')` mesh with default `AxisType.Auto` axes (no `AxisType.Explicit` model rewrite required) and construct a Transformer parameter PyTree covering `2D` matrices, `3D` multi-head attention tensors, and `1D` layer-norm scales.

In [2]:
# 2D ('data', 'model') mesh with default AxisType.Auto
auto_mesh = jax.make_mesh(
    (2, 4), ('data', 'model'), axis_types=(AxisType.Auto, AxisType.Auto)
)

d_model, d_ff, vocab, n_heads, head_dim = 512, 2048, 4096, 8, 64
abstract_params = {
    'embed': jax.ShapeDtypeStruct(
        (vocab, d_model), jnp.float32, sharding=NamedSharding(auto_mesh, P('data', 'model'))
    ),
    'attn_qkv': jax.ShapeDtypeStruct(
        (d_model, n_heads, 3 * head_dim),
        jnp.float32,
        sharding=NamedSharding(auto_mesh, P('data', 'model', None)),
    ),
    'attn_out': jax.ShapeDtypeStruct(
        (n_heads, head_dim, d_model),
        jnp.float32,
        sharding=NamedSharding(auto_mesh, P('model', None, 'data')),
    ),
    'mlp_up': jax.ShapeDtypeStruct(
        (d_model, d_ff), jnp.float32, sharding=NamedSharding(auto_mesh, P('data', 'model'))
    ),
    'mlp_down': jax.ShapeDtypeStruct(
        (d_ff, d_model), jnp.float32, sharding=NamedSharding(auto_mesh, P('model', 'data'))
    ),
    'norm_scale': jax.ShapeDtypeStruct(
        (d_model,), jnp.float32, sharding=NamedSharding(auto_mesh, P('model',))
    ),
}

data_sharding = NamedSharding(auto_mesh, P('data', None))
abstract_batch = {
    'ids': jax.ShapeDtypeStruct((8, 128), jnp.int32, sharding=data_sharding),
    'targets': jax.ShapeDtypeStruct((8, 128), jnp.int32, sharding=data_sharding),
}


def loss_fn(params, batch):
  x = params['embed'][batch['ids']] * params['norm_scale']
  qkv = jnp.einsum('btd,dhm->bthm', x, params['attn_qkv'])
  q, k, v = jnp.split(qkv, 3, axis=-1)
  attn = jax.nn.softmax(jnp.einsum('bthd,bshd->bhts', q, k) / math.sqrt(head_dim))
  ctx = jnp.einsum('bhts,bshd->bthd', attn, v)
  x = x + jnp.einsum('bthd,hdm->btm', ctx, params['attn_out'])
  h = jax.nn.gelu(jnp.einsum('btd,df->btf', x, params['mlp_up']))
  x = x + jnp.einsum('btf,fd->btd', h, params['mlp_down'])
  logits = jnp.einsum('btd,vd->btv', x, params['embed'])
  log_probs = jax.nn.log_softmax(logits, axis=-1)
  return -jnp.mean(jnp.take_along_axis(log_probs, batch['targets'][..., None], axis=-1))


for k, v in abstract_params.items():
  print(f'{k:12s}: shape={str(v.shape):16s} sharding={v.sharding.spec}')

embed       : shape=(4096, 512)      sharding=P('data', 'model')
attn_qkv    : shape=(512, 8, 192)    sharding=P('data', 'model', None)
attn_out    : shape=(8, 64, 512)     sharding=P('model', None, 'data')
mlp_up      : shape=(512, 2048)      sharding=P('data', 'model')
mlp_down    : shape=(2048, 512)      sharding=P('model', 'data')
norm_scale  : shape=(512,)           sharding=P('model',)


## 2: `jax.eval_shape` vs. `optax.tree_utils.abstract_init`

When we lower and compile `train_step` (`donate_argnums=(0,)`, `out_shardings=...`) from abstract inputs alone, XLA's `compiled.memory_analysis()` reports exact per-device buffer bytes (`argument_size_in_bytes`, `alias_size_in_bytes`, `temp_size_in_bytes`). Notice how `jax.eval_shape` + replicating `None` leaves inflates `argument_size_in_bytes` from **`7.51 MiB/chip` to `42.51 MiB/chip`** and doubles total step peak memory.

In [3]:
def compile_step_and_analyze(
    tx: optax.GradientTransformation,
    abstract_opt_state: Any,
    mesh: jax.sharding.Mesh,
):
  """Lowers & compiles train_step with donate_argnums=(0,) and returns XLA memory stats."""
  abstract_state = {'params': abstract_params, 'opt_state': abstract_opt_state}
  state_shardings = jax.tree.map(lambda l: l.sharding, abstract_state)
  replicated = NamedSharding(mesh, P())

  def train_step(state, batch):
    loss, grads = jax.value_and_grad(loss_fn)(state['params'], batch)
    updates, new_opt = tx.update(grads, state['opt_state'], state['params'])
    new_params = optax.apply_updates(state['params'], updates)
    return {'params': new_params, 'opt_state': new_opt}, loss

  compiled = (
      jax.jit(
          train_step,
          donate_argnums=(0,),
          out_shardings=(state_shardings, replicated),
      )
      .lower(abstract_state, abstract_batch)
      .compile()
  )
  stats = compiled.memory_analysis()
  arg_mib = stats.argument_size_in_bytes / 1024**2
  alias_mib = stats.alias_size_in_bytes / 1024**2
  temp_mib = stats.temp_size_in_bytes / 1024**2
  peak_mib = (
      stats.argument_size_in_bytes
      + stats.output_size_in_bytes
      - stats.alias_size_in_bytes
      + stats.temp_size_in_bytes
  ) / 1024**2
  gflops = compiled.cost_analysis().get('flops', 0.0) / 1e9
  return compiled, arg_mib, alias_mib, temp_mib, peak_mib, gflops


tx_adamw = optax.adamw(1e-3)
replicated = NamedSharding(auto_mesh, P())

# 1. Naive jax.eval_shape + default None leaves to replicated P()
naive_opt_state = jax.tree.map(
    lambda l: jax.ShapeDtypeStruct(l.shape, l.dtype, sharding=l.sharding or replicated),
    jax.eval_shape(tx_adamw.init, abstract_params),
)
_, n_arg, n_alias, n_temp, n_peak, _ = compile_step_and_analyze(
    tx_adamw, naive_opt_state, auto_mesh
)

# 2. optax.tree_utils.abstract_init
sharded_opt_state = optax.tree_utils.abstract_init(tx_adamw, abstract_params)
_, s_arg, s_alias, s_temp, s_peak, _ = compile_step_and_analyze(
    tx_adamw, sharded_opt_state, auto_mesh
)

print(f'BEFORE (Naive eval_shape + P())        : state+batch={n_arg:6.2f} MiB | donated={n_alias:6.2f} MiB | peak={n_peak:6.2f} MiB/chip')
print(f'AFTER  (optax.tree_utils.abstract_init): state+batch={s_arg:6.2f} MiB | donated={s_alias:6.2f} MiB | peak={s_peak:6.2f} MiB/chip')

BEFORE (Naive eval_shape + P())        : state+batch= 42.51 MiB | donated= 42.50 MiB | peak=106.02 MiB/chip
AFTER  (optax.tree_utils.abstract_init): state+batch=  7.51 MiB | donated=  7.50 MiB | peak= 52.03 MiB/chip


`42.50 MiB` (from `jax.eval_shape`) vs. `7.50 MiB` (from `optax.tree_utils.abstract_init`):

Our simple Transformer has **`20.00 MiB` of `float32` weights globally** across the 8 chips (`2.50 MiB/chip` when sharded):

- **Naive `jax.eval_shape` + `P()`:**
  - `params` (sharded $\div 8$): **`2.50 MiB/chip`**
  - `mu` (replicated `P()` on every chip): **`20.00 MiB/chip`**
  - `nu` (replicated `P()` on every chip): **`20.00 MiB/chip`**
  - **Total `state` = `42.50 MiB/chip`** (`+ 0.01 MiB batch = 42.51 MiB`), and XLA donates all `42.50 MiB` of this in-place.
- **`optax.tree_utils.abstract_init`:**
  - `params` (sharded $\div 8$): **`2.50 MiB/chip`**
  - `mu` (sharded $\div 8$): **`2.50 MiB/chip`**
  - `nu` (sharded $\div 8$): **`2.50 MiB/chip`**
  - **Total `state` = `7.50 MiB/chip`** (`+ 0.01 MiB batch = 7.51 MiB`), and because `optax.tree_utils.abstract_init`'s `out_shardings` match `in_shardings`, XLA donates **(`7.50 / 7.50 MiB`)** of the sharded state in-place.